# Gaussian bandwidth-choice distributions: exponential symmetric divergences

Compare **L², TV, H², JS, and the sample-specific ISE choice** on the same repeated samples from $N(0,1)$ and the same geometric bandwidth grid.
For every spectral method, use $S_{ij}=\exp(-\lambda D_{ij})$ with **$\lambda=1$** and select the bandwidth at the largest Perron-vector component.
The ISE choice minimizes exact integrated squared error over the same candidate grid.

| Method | Divergence used in the exponential | Calculation |
|---|---|---|
| L² | $\frac12\int(p-q)^2$ | Exact Gaussian KDE inner products |
| TV | $\frac12\int|p-q|$ | Composite trapezoidal integration |
| H² | $\frac12\int(\sqrt p-\sqrt q)^2$ | Composite trapezoidal integration |
| JS | $\frac12\operatorname{KL}(p\Vert(p+q)/2)+\frac12\operatorname{KL}(q\Vert(p+q)/2)$ | Composite trapezoidal integration; natural logarithms |

The L² factor $1/2$ matches the thesis's bandwidth-distribution experiment. TV, H² and JS enter the exponential as **raw divergences**, without conversion to overlap/affinity similarities or JS division by $\log 2$.
Sources: [thesis divergence definitions and quadrature subsection](../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex), especially the Hellinger definition, `eq:JS_KL_formula`, and `eq:beta2_exact_formula01`; [shared exact Gaussian implementation](kde_tools/similarities.py); [shared composite trapezoidal implementation](kde_tools/quadrature.py).

Run all cells. The **preview** profile produces a small comparison; **thesis** uses the original Figure 3.5 sample/repetition/grid settings. A checkpoint saves each completed repetition, so rerunning resumes the same configuration.
All spectral choices use the centralized stabilized Lanczos solver in `kde_tools.perron`; checkpoint settings record its version.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import importlib
import json
import math
import os
import sys
import time
from pathlib import Path

working_directory = Path.cwd().resolve()
code_directory = next(
    (candidate.resolve()
     for root in (working_directory, *working_directory.parents)
     for candidate in (root, root / "Code" / "Code_kde")
     if (candidate / "kde_tools" / "pdf_export.py").is_file()),
    None,
)
if code_directory is None:
    raise FileNotFoundError("Run this notebook from the project or Code/Code_kde folder.")
if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))
os.environ.setdefault("MPLCONFIGDIR", str(code_directory / ".matplotlib-cache"))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import PercentFormatter

from kde_tools.bandwidths import bandwidth_grid_linear_in_psi_beta, gaussian_mise_oracle_bandwidth
from kde_tools.density import exact_oracle_ise_values, gaussian_kde_values
from kde_tools.perron import PERRON_SOLVER_VERSION
from kde_tools.pdf_export import enable_automatic_pdf_saving
# Refresh source changes when rerunning setup in an existing notebook kernel.
# See https://docs.python.org/3/library/importlib.html#importlib.reload.
import kde_tools.quadrature as _quadrature
importlib.reload(_quadrature)
from kde_tools.quadrature import quadrature_js_divergence_matrix, quadrature_grid_for_kdes, trapezoidal_integral
from kde_tools.similarities import beta2_divergence_matrix, spectral_choice_from_divergence

notebook_name = "gaussian_symmetric_divergence_bandwidth_distributions"
pdf_output_directory = enable_automatic_pdf_saving(
    notebook_name, output_root=code_directory / "outputs" / "pdfs"
)
data_output_directory = code_directory / "outputs" / "data" / notebook_name
image_output_directory = code_directory / "outputs" / "images" / notebook_name
for directory in (data_output_directory, image_output_directory):
    directory.mkdir(parents=True, exist_ok=True)
print(f"Automatic PDF exports: {pdf_output_directory}")


## Settings

Both profiles draw $n=200$ observations from $N(0,1)$ and use $\lambda=1$, $h_{\min}=0.01$, $h_{\max}=2$. The preview uses $M=20$, $N=21$; the thesis profile uses $M=1000$, $N=201$, matching the [Figure 3.5 caption](../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex).
`m_x` is the number of shared spatial nodes for TV, H², and JS. All settings are editable below.

The interval $[X_{\min}-Lh_{\max},X_{\max}+Lh_{\max}]$ follows the thesis's truncation construction. The Gaussian tail-mass bound $\delta_L=2\Phi(-L)$ also bounds the omitted tails of the TV and H² integrands: pointwise, each is at most $(p+q)/2$ (direct deduction from their definitions). This controls **tail truncation only**. Spatial discretization errors still require refinement checks; no total error certificate is claimed.
Sources: [thesis quadrature subsection, truncation bounds](../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [official composite trapezoidal documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.trapezoid.html).

In [ ]:
RUN_PROFILE = "thesis"  # "preview" or "thesis"
PROFILES = {
    "preview": {"sample_size": 200, "repetitions": 20, "grid_size": 21},
    "thesis": {"sample_size": 200, "repetitions": 1000, "grid_size": 201},
}
settings = {
    "perron_solver": PERRON_SOLVER_VERSION,
    **PROFILES[RUN_PROFILE],
    "sigma2": 1.0,
    "lambda_value": 1.0,
    "h_min": 0.01,
    "h_max": 2.0,
    "m_x": 6001,
    "padding": 6.0,
    "base_seed": 910_000,
    "implementation_version": 2,
    "js_quadrature": "composite_trapezoidal",
}
reuse_checkpoint = True
show_mise_reference = True

if settings["sample_size"] < 1 or settings["repetitions"] < 1:
    raise ValueError("sample_size and repetitions must be positive.")
if settings["sigma2"] <= 0 or settings["lambda_value"] <= 0:
    raise ValueError("sigma2 and lambda_value must be positive.")
h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=settings["h_min"], h_max=settings["h_max"],
    beta=2.0, grid_size=settings["grid_size"],
)
methods = ("L2", "TV", "H2", "JS", "ISE")
configuration_json = json.dumps(settings, sort_keys=True)
configuration_id = hashlib.sha256(configuration_json.encode()).hexdigest()[:12]
run_directory = data_output_directory / RUN_PROFILE / configuration_id
run_directory.mkdir(parents=True, exist_ok=True)
checkpoint_path = run_directory / "checkpoint.npz"
plot_stem = f"{RUN_PROFILE}_n{settings['sample_size']}_M{settings['repetitions']}_N{len(h_grid)}"
tail_mass_bound = math.erfc(settings["padding"] / math.sqrt(2.0))
print(f"Profile: {RUN_PROFILE}; n={settings['sample_size']}, M={settings['repetitions']}, N={len(h_grid)}, lambda={settings['lambda_value']:g}")
print(f"TV/H² tail truncation bound: {tail_mass_bound:.3e}; spatial nodes: {settings['m_x']}")
print("JS uses composite trapezoidal integration on the same grid as TV/H².")
print(f"Checkpoint and data: {run_directory}")

## One repetition: four exponential spectral choices and one ISE choice

The same sample supplies all five choices. L² uses the shared exact divergence. TV, H², and JS integrate the original KDEs over the same truncated interval and reuse a single density evaluation, without renormalizing the densities on that interval. JS uses the shared composite trapezoidal divergence routine with natural logarithms.
The H² calculation below is the algebraic expansion of $\frac12\int(\sqrt p-\sqrt q)^2$; the trapezoidal weights are $\Delta x/2$ at the endpoints and $\Delta x$ inside. Sources: [thesis composite trapezoidal rule](../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex), [official trapezoidal integration documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.trapezoid.html).

In [ ]:
def compute_divergences(samples, h_grid, *, m_x, padding):
    """Build raw L², TV, H², JS matrices for the same Gaussian KDE family."""
    x_grid = quadrature_grid_for_kdes(samples, h_grid, m_x=m_x, padding=padding)
    densities = gaussian_kde_values(samples, h_grid, x_grid)
    masses = trapezoidal_integral(densities, x_grid, axis=1)
    number_of_bandwidths = len(h_grid)

    tv = np.zeros((number_of_bandwidths, number_of_bandwidths))
    for i in range(number_of_bandwidths - 1):
        values = 0.5 * trapezoidal_integral(
            np.abs(densities[i + 1:] - densities[i]), x_grid, axis=1
        )
        tv[i, i + 1:] = values
        tv[i + 1:, i] = values

    weights = np.full(len(x_grid), (x_grid[-1] - x_grid[0]) / (len(x_grid) - 1))
    weights[[0, -1]] *= 0.5
    square_roots = np.sqrt(densities)
    affinity = (square_roots * weights[None, :]) @ square_roots.T
    h_squared = np.maximum(0.5 * (masses[:, None] + masses[None, :]) - affinity, 0.0)

    divergences = {
        "L2": beta2_divergence_matrix(samples, h_grid),
        "TV": tv,
        "H2": h_squared,
        "JS": quadrature_js_divergence_matrix(densities, x_grid),
    }
    for name, divergence in divergences.items():
        divergence[:] = 0.5 * (divergence + divergence.T)
        np.fill_diagonal(divergence, 0.0)
        if not np.all(np.isfinite(divergence)) or np.any(divergence < 0):
            raise ArithmeticError(f"Invalid {name} divergence matrix.")
    diagnostics = {"maximum_grid_mass_error": float(np.max(np.abs(masses - 1.0)))}
    return divergences, diagnostics


def select_one_sample(samples, h_grid, settings):
    divergences, diagnostics = compute_divergences(
        samples, h_grid, m_x=settings["m_x"], padding=settings["padding"],
    )
    selections = {}
    for name, divergence in divergences.items():
        selections[name] = int(spectral_choice_from_divergence(
            divergence, settings["lambda_value"]
        )["selected_index"])
    ise_values = exact_oracle_ise_values(samples, h_grid, settings["sigma2"])
    selections["ISE"] = int(np.argmin(ise_values))
    selected_ise = {name: float(ise_values[index]) for name, index in selections.items()}
    return selections, selected_ise, diagnostics

## Repeated sampling, with resumable checkpoints

Each repetition uses seed `base_seed + repetition`, independently of which methods are computed. The checkpoint contains the settings and all completed five-method choices. Changing numerical settings creates a new run directory. Set `reuse_checkpoint=False` to recompute the current configuration.

In [ ]:
repetitions = settings["repetitions"]
selected_indices = np.full((repetitions, len(methods)), -1, dtype=int)
selected_ise_values = np.full((repetitions, len(methods)), np.nan)
grid_mass_errors = np.full(repetitions, np.nan)
completed = 0
if reuse_checkpoint and checkpoint_path.exists():
    with np.load(checkpoint_path, allow_pickle=False) as checkpoint:
        if str(checkpoint["configuration_json"].item()) != configuration_json:
            raise ValueError("Checkpoint settings differ from the current configuration.")
        completed = int(checkpoint["completed"].item())
        selected_indices[:] = checkpoint["selected_indices"]
        selected_ise_values[:] = checkpoint["selected_ise_values"]
        grid_mass_errors[:] = checkpoint["grid_mass_errors"]
    if not 0 <= completed <= repetitions or np.any(selected_indices[:completed] < 0):
        raise ValueError("Checkpoint has invalid completed selections.")
    print(f"Resuming after {completed}/{repetitions} completed repetitions.")

started = time.perf_counter()
for repetition in range(completed, repetitions):
    rng = np.random.default_rng(settings["base_seed"] + repetition)
    samples = rng.normal(0.0, np.sqrt(settings["sigma2"]), size=settings["sample_size"])
    choices, ise_values, diagnostics = select_one_sample(samples, h_grid, settings)
    selected_indices[repetition] = [choices[name] for name in methods]
    selected_ise_values[repetition] = [ise_values[name] for name in methods]
    grid_mass_errors[repetition] = diagnostics["maximum_grid_mass_error"]
    temporary_checkpoint = checkpoint_path.with_suffix(".npz.tmp")
    with temporary_checkpoint.open("wb") as stream:
        np.savez_compressed(
            stream, configuration_json=configuration_json, completed=repetition + 1,
            selected_indices=selected_indices, selected_ise_values=selected_ise_values,
            grid_mass_errors=grid_mass_errors,
        )
    temporary_checkpoint.replace(checkpoint_path)
    print(f"Completed {repetition + 1}/{repetitions}; elapsed this run: {time.perf_counter() - started:.1f}s", flush=True)

counts = {name: np.bincount(selected_indices[:, column], minlength=len(h_grid))
          for column, name in enumerate(methods)}
frequencies = {name: values / repetitions for name, values in counts.items()}
for name in methods:
    assert counts[name].sum() == repetitions
    assert np.isclose(frequencies[name].sum(), 1.0)
assert np.all(selected_ise_values[:, -1, None] <= selected_ise_values + 1e-12)
print(f"Maximum observed grid mass error: {np.max(grid_mass_errors):.3e}")

## Summary and saved data

The table reports selected bandwidths and the exact ISE evaluated at each choice. Export the individual choices, frequency table, summary, and settings so that the plotted distributions can be reproduced without rerunning the integrations.

In [ ]:
def write_csv(path, fieldnames, rows):
    with path.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

selection_rows = [
    {"repetition": repetition, "seed": settings["base_seed"] + repetition,
     "method": name, "grid_index": int(selected_indices[repetition, column]),
     "h": float(h_grid[selected_indices[repetition, column]]),
     "ise": float(selected_ise_values[repetition, column])}
    for repetition in range(repetitions) for column, name in enumerate(methods)
]
frequency_rows = [
    {"method": name, "grid_index": index, "h": float(h_grid[index]),
     "count": int(counts[name][index]), "relative_frequency": float(frequencies[name][index])}
    for name in methods for index in range(len(h_grid))
]
summary_rows = []
for column, name in enumerate(methods):
    selected_h = h_grid[selected_indices[:, column]]
    modal_index = int(np.argmax(counts[name]))
    summary_rows.append({
        "method": name, "modal_h": float(h_grid[modal_index]),
        "mean_h": float(np.mean(selected_h)), "median_h": float(np.median(selected_h)),
        "modal_frequency": float(frequencies[name][modal_index]),
        "mean_ise": float(np.mean(selected_ise_values[:, column])),
    })
write_csv(run_directory / "selections.csv", selection_rows[0].keys(), selection_rows)
write_csv(run_directory / "frequencies.csv", frequency_rows[0].keys(), frequency_rows)
write_csv(run_directory / "summary.csv", summary_rows[0].keys(), summary_rows)
(run_directory / "settings.json").write_text(json.dumps({
    "profile": RUN_PROFILE, "settings": settings, "methods": list(methods),
    "tail_mass_bound": tail_mass_bound, "maximum_grid_mass_error": float(np.max(grid_mass_errors)),
}, indent=2) + "\n")
print(f"{'method':8s} {'modal h':>10s} {'mean h':>10s} {'median h':>10s} {'mean ISE':>12s}")
for row in summary_rows:
    print(f"{row['method']:8s} {row['modal_h']:10.5f} {row['mean_h']:10.5f} {row['median_h']:10.5f} {row['mean_ise']:12.6f}")
print(f"CSV data and settings saved to {run_directory}")

## Bandwidth-choice distribution figure

Overlaid bars show the five relative-frequency distributions. As in Figure 3.5, the horizontal positions are equally spaced geometric-grid indices, labeled with their bandwidth values. The optional dashed line shows the Gaussian population MISE optimum; the purple bars show the **sample-specific ISE choice**.
The title records the actual $n$, $M$, $N$, and $\lambda$, so a preview is distinguishable from the larger simulation. Every displayed figure is automatically exported to PDF; a PNG is also saved for inspection.
Source for the optional reference: [exact Gaussian MISE helper](kde_tools/bandwidths.py).

In [ ]:
colors = {"L2": "#D6843B", "TV": "#0072B2", "H2": "#49A994", "JS": "#C44E52", "ISE": "#9B73C5"}
labels = {"L2": r"$L^2$ spectral", "TV": "TV spectral", "H2": r"$H^2$ spectral", "JS": "JS spectral", "ISE": "ISE-optimal grid bandwidths"}
grid_positions = np.arange(len(h_grid))
fig, ax = plt.subplots(figsize=(13.5, 6.5))
for name in methods:
    ax.bar(grid_positions, frequencies[name], width=0.86, color=colors[name],
           alpha=0.48, edgecolor="white", linewidth=0.35, label=labels[name])
if show_mise_reference:
    mise_bandwidth = np.sqrt(settings["sigma2"]) * gaussian_mise_oracle_bandwidth(settings["sample_size"])
    if h_grid[0] <= mise_bandwidth <= h_grid[-1]:
        mise_position = np.interp(np.log(mise_bandwidth), np.log(h_grid), grid_positions)
        ax.axvline(mise_position, color="#1f77b4", linestyle="--", linewidth=2.0,
                   label=f"Gaussian MISE optimum: h={mise_bandwidth:.3f}")
ticks = np.unique(np.linspace(0, len(h_grid) - 1, 7, dtype=int))
ax.set_xticks(ticks, [f"{h_grid[index]:.3g}" for index in ticks])
ax.set_xlim(-0.7, len(h_grid) - 0.3)
ax.set_ylim(0, 1.08 * max(np.max(values) for values in frequencies.values()))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
ax.set_xlabel("selected bandwidth h", fontsize=13)
ax.set_ylabel("relative frequency", fontsize=13)
ax.set_title(
    "Bandwidth-choice distributions: Gaussian data\n"
    + rf"$n={settings['sample_size']}$, $M={repetitions}$, $N={len(h_grid)}$, "
    + rf"$S=\exp(-\lambda D)$, $\lambda={settings['lambda_value']:g}$ ({RUN_PROFILE})",
    fontsize=15, pad=18,
)
ax.grid(axis="y", color="#DDE5EC", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#B8C5D1")
ax.tick_params(labelsize=11, colors="#304967")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False, fontsize=10.5)
fig.tight_layout()
png_output_path = image_output_directory / f"{plot_stem}.png"
fig.savefig(png_output_path, dpi=160, bbox_inches="tight")
plt.show()  # The shared automatic exporter saves the PDF before display.
print(f"PNG: {png_output_path}")
print(f"PDF directory: {pdf_output_directory}")

## Separate bandwidth-choice distribution plots

Each distribution from the overlaid figure above is plotted again in its own figure, in the same method order and colors. All five plots use the same bandwidth positions and vertical limits as the overlaid figure. They reuse the existing `frequencies`; no new simulation is run.

The optional Gaussian MISE reference is repeated in each plot. Each figure is displayed below and exported as a PNG and, through the shared automatic exporter, a PDF.


In [ ]:
# Reuse the distributions and colors from the overlaid figure above.
separate_grid_positions = np.arange(len(h_grid))
separate_ticks = np.unique(np.linspace(0, len(h_grid) - 1, 7, dtype=int))
separate_y_max = 1.08 * max(np.max(values) for values in frequencies.values())
separate_mise_position = None
if show_mise_reference:
    separate_mise_bandwidth = np.sqrt(settings["sigma2"]) * gaussian_mise_oracle_bandwidth(settings["sample_size"])
    if h_grid[0] <= separate_mise_bandwidth <= h_grid[-1]:
        separate_mise_position = np.interp(
            np.log(separate_mise_bandwidth), np.log(h_grid), separate_grid_positions
        )

for name in methods:
    separate_fig, separate_ax = plt.subplots(figsize=(13.5, 6.5))
    separate_ax.bar(
        separate_grid_positions, frequencies[name], width=0.86,
        color=colors[name], alpha=0.85, edgecolor="white", linewidth=0.35,
        label=labels[name],
    )
    if separate_mise_position is not None:
        separate_ax.axvline(
            separate_mise_position, color="#1f77b4", linestyle="--", linewidth=2.0,
            label=f"Gaussian MISE optimum: h={separate_mise_bandwidth:.3f}",
        )
    separate_ax.set_xticks(
        separate_ticks, [f"{h_grid[index]:.3g}" for index in separate_ticks]
    )
    separate_ax.set_xlim(-0.7, len(h_grid) - 0.3)
    separate_ax.set_ylim(0, separate_y_max)
    separate_ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    separate_ax.set_xlabel("selected bandwidth h", fontsize=13)
    separate_ax.set_ylabel("relative frequency", fontsize=13)
    separate_ax.set_title(
        f"Bandwidth-choice distribution: {labels[name]}\n"
        + rf"Gaussian data: $n={settings['sample_size']}$, $M={repetitions}$, "
        + rf"$N={len(h_grid)}$, $\lambda={settings['lambda_value']:g}$ ({RUN_PROFILE})",
        fontsize=15, pad=18,
    )
    separate_ax.grid(axis="y", color="#DDE5EC", linewidth=0.8)
    separate_ax.set_axisbelow(True)
    for side in ("top", "right"):
        separate_ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        separate_ax.spines[side].set_color("#B8C5D1")
    separate_ax.tick_params(labelsize=11, colors="#304967")
    separate_ax.legend(
        loc="upper center", bbox_to_anchor=(0.5, -0.15),
        ncol=2, frameon=False, fontsize=10.5,
    )
    separate_fig.tight_layout()
    separate_png_output_path = image_output_directory / f"{plot_stem}_{name.lower()}.png"
    separate_fig.savefig(separate_png_output_path, dpi=160, bbox_inches="tight")
    plt.show()  # The shared automatic exporter saves this figure as a PDF.
    plt.close(separate_fig)
    print(f"PNG ({name}): {separate_png_output_path}")

print(f"PDF directory: {pdf_output_directory}")


## Figure 3.6 extension: TV, H², and JS for increasing sample size

Use **ten sample sizes** $n=100,200,\ldots,1000$. Each divergence gets its own figure with **ten separate distribution panels**, arranged in increasing $n$. Keep the Gaussian data, fixed geometric bandwidth grid, and $S_{ij}=\exp(-\lambda D_{ij})$ with $\lambda=1$ from the experiment above. The default thesis profile uses $M=1000$ repetitions at each $n$ and $N=201$ candidate bandwidths; the preview profile uses $M=20$ and $N=21$.

Within a repetition, the three divergences use the same sample and KDE evaluations. Reusing the seed `base_seed + repetition` across sample sizes gives nested Gaussian sample prefixes. TV, H², and JS use the same composite trapezoidal integration as above; only these three divergences are computed. The settings remain separate from the earlier experiment.

Checkpoints are saved after every repetition, separately for each $n$. An existing single-size checkpoint is reused only when its settings and stored method names match. In particular, the completed $n=200$ thesis run can be reused. The final plotting cell uses common horizontal and vertical scales across all 30 panels and automatically saves each displayed figure as a PDF; PNGs and CSV data are also saved.

Run the cells in this section in order after the notebook setup and settings. The full thesis run computes the remaining sample sizes; switch `N_SWEEP_PROFILE` to `"preview"` for a shorter exploratory run. Preview results use a different repetition count and bandwidth grid.

Sources: [Figure 3.6 / increasing-sample-size experiment, `fig:distribution_of_chosen_bandwidths_for_increasing_sample_size`](../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [divergence definitions, Hellinger definition and `eq:JS_KL_formula`](../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [shared quadrature and JS implementation](kde_tools/quadrature.py); [shared spectral selector](kde_tools/similarities.py).


In [ ]:
N_SWEEP_PROFILE = RUN_PROFILE  # "thesis" or "preview"
N_SWEEP_SAMPLE_SIZES = list(range(100, 1001, 100))
N_SWEEP_METHODS = ("TV", "H2", "JS")
N_SWEEP_REUSE_CHECKPOINTS = True
N_SWEEP_SHOW_MISE_REFERENCE = show_mise_reference

n_sweep_base_settings = {key: value for key, value in settings.items() if key != "sample_size"}
n_sweep_base_settings.update(
    repetitions=PROFILES[N_SWEEP_PROFILE]["repetitions"],
    grid_size=PROFILES[N_SWEEP_PROFILE]["grid_size"],
    lambda_value=1.0,
)
n_sweep_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=n_sweep_base_settings["h_min"], h_max=n_sweep_base_settings["h_max"],
    beta=2.0, grid_size=n_sweep_base_settings["grid_size"],
)
n_sweep_configuration = {
    "settings": n_sweep_base_settings,
    "sample_sizes": N_SWEEP_SAMPLE_SIZES,
    "methods": list(N_SWEEP_METHODS),
    "sweep_implementation_version": 1,
    "sampling": "normal_prefixes_from_base_seed_plus_repetition",
}
n_sweep_configuration_id = hashlib.sha256(
    json.dumps(n_sweep_configuration, sort_keys=True).encode()
).hexdigest()[:12]
n_sweep_run_directory = (
    data_output_directory / "increasing_sample_size" / N_SWEEP_PROFILE / n_sweep_configuration_id
)
n_sweep_image_directory = (
    image_output_directory / "increasing_sample_size" / N_SWEEP_PROFILE / n_sweep_configuration_id
)
for directory in (n_sweep_run_directory, n_sweep_image_directory):
    directory.mkdir(parents=True, exist_ok=True)
(n_sweep_run_directory / "settings.json").write_text(
    json.dumps({"profile": N_SWEEP_PROFILE, **n_sweep_configuration}, indent=2) + "\n"
)
print(f"Sample sizes: {N_SWEEP_SAMPLE_SIZES}; methods: {N_SWEEP_METHODS}")
print(f"Profile: {N_SWEEP_PROFILE}; M={n_sweep_base_settings['repetitions']} per n; "
      f"N={len(n_sweep_h_grid)}; lambda={n_sweep_base_settings['lambda_value']:g}")
print(f"Checkpoints and CSV data: {n_sweep_run_directory}")


In [ ]:
def compute_n_sweep_divergences(samples, h_grid, *, m_x, padding):
    """Compute raw TV, H², and JS on one shared trapezoidal KDE grid."""
    x_grid = quadrature_grid_for_kdes(samples, h_grid, m_x=m_x, padding=padding)
    densities = gaussian_kde_values(samples, h_grid, x_grid)
    masses = trapezoidal_integral(densities, x_grid, axis=1)
    number_of_bandwidths = len(h_grid)

    tv = np.zeros((number_of_bandwidths, number_of_bandwidths))
    for i in range(number_of_bandwidths - 1):
        values = 0.5 * trapezoidal_integral(
            np.abs(densities[i + 1:] - densities[i]), x_grid, axis=1
        )
        tv[i, i + 1:] = values
        tv[i + 1:, i] = values

    weights = np.full(len(x_grid), (x_grid[-1] - x_grid[0]) / (len(x_grid) - 1))
    weights[[0, -1]] *= 0.5
    square_roots = np.sqrt(densities)
    affinity = (square_roots * weights[None, :]) @ square_roots.T
    h_squared = np.maximum(0.5 * (masses[:, None] + masses[None, :]) - affinity, 0.0)
    divergences = {
        "TV": tv,
        "H2": h_squared,
        "JS": quadrature_js_divergence_matrix(densities, x_grid),
    }
    for name, divergence in divergences.items():
        divergence[:] = 0.5 * (divergence + divergence.T)
        np.fill_diagonal(divergence, 0.0)
        if not np.all(np.isfinite(divergence)) or np.any(divergence < 0):
            raise ArithmeticError(f"Invalid {name} divergence matrix.")
    return divergences, float(np.max(np.abs(masses - 1.0)))


def _validate_n_sweep_checkpoint(indices, mass_errors, completed, repetitions, grid_size):
    if indices.shape != (repetitions, len(N_SWEEP_METHODS)) or mass_errors.shape != (repetitions,):
        raise ValueError("Checkpoint array shapes differ from the current experiment.")
    if not 0 <= completed <= repetitions:
        raise ValueError("Invalid checkpoint completion count.")
    if np.any(indices[:completed] < 0) or np.any(indices[:completed] >= grid_size):
        raise ValueError("Checkpoint contains invalid selected bandwidth indices.")
    if not np.all(np.isfinite(mass_errors[:completed])):
        raise ValueError("Checkpoint contains invalid quadrature diagnostics.")


def run_n_sweep_point(sample_size, base_settings, h_grid, output_directory, *,
                      existing_data_directory=None, profile="thesis", reuse=True):
    """Run or resume the three paired selectors for one sample size."""
    point_settings = {**base_settings, "sample_size": int(sample_size)}
    repetitions = int(point_settings["repetitions"])
    if repetitions < 1 or sample_size < 1:
        raise ValueError("sample_size and repetitions must be positive.")
    point_configuration = json.dumps({
        "settings": point_settings, "methods": list(N_SWEEP_METHODS),
        "sweep_implementation_version": 1,
    }, sort_keys=True)
    point_directory = Path(output_directory) / f"n{int(sample_size)}"
    point_directory.mkdir(parents=True, exist_ok=True)
    checkpoint_path = point_directory / "checkpoint.npz"
    indices = np.full((repetitions, len(N_SWEEP_METHODS)), -1, dtype=int)
    mass_errors = np.full(repetitions, np.nan)
    completed = 0
    source_checkpoint = None

    def save_checkpoint():
        temporary_path = checkpoint_path.with_suffix(".npz.tmp")
        with temporary_path.open("wb") as stream:
            np.savez_compressed(
                stream, configuration_json=point_configuration, h_grid=h_grid,
                selected_indices=indices, grid_mass_errors=mass_errors, completed=completed,
            )
        temporary_path.replace(checkpoint_path)

    if reuse and checkpoint_path.exists():
        with np.load(checkpoint_path, allow_pickle=False) as checkpoint:
            if str(checkpoint["configuration_json"].item()) != point_configuration:
                raise ValueError("Sweep checkpoint settings differ from the current configuration.")
            if not np.array_equal(checkpoint["h_grid"], h_grid):
                raise ValueError("Sweep checkpoint bandwidth grid differs from the current grid.")
            indices = checkpoint["selected_indices"].copy()
            mass_errors = checkpoint["grid_mass_errors"].copy()
            completed = int(checkpoint["completed"].item())
    elif reuse and existing_data_directory is not None:
        # Reuse the earlier single-size experiment only after exact settings checks.
        source_json = json.dumps(point_settings, sort_keys=True)
        source_id = hashlib.sha256(source_json.encode()).hexdigest()[:12]
        source_directory = Path(existing_data_directory) / profile / source_id
        source_path = source_directory / "checkpoint.npz"
        source_manifest_path = source_directory / "settings.json"
        if source_path.exists() and source_manifest_path.exists():
            source_manifest = json.loads(source_manifest_path.read_text())
            source_methods = source_manifest.get("methods", [])
            if (source_manifest.get("settings") == point_settings
                    and all(name in source_methods for name in N_SWEEP_METHODS)):
                with np.load(source_path, allow_pickle=False) as checkpoint:
                    if str(checkpoint["configuration_json"].item()) != source_json:
                        raise ValueError("Existing single-size checkpoint settings are inconsistent.")
                    source_indices = checkpoint["selected_indices"]
                    if source_indices.shape != (repetitions, len(source_methods)):
                        raise ValueError("Existing single-size checkpoint has invalid array shapes.")
                    columns = [source_methods.index(name) for name in N_SWEEP_METHODS]
                    indices = source_indices[:, columns].copy()
                    mass_errors = checkpoint["grid_mass_errors"].copy()
                    completed = int(checkpoint["completed"].item())
                source_checkpoint = str(source_path)

    _validate_n_sweep_checkpoint(indices, mass_errors, completed, repetitions, len(h_grid))
    if source_checkpoint is not None:
        save_checkpoint()
        print(f"n={sample_size}: reused {completed}/{repetitions} repetitions from {source_checkpoint}", flush=True)
    else:
        print(f"n={sample_size}: starting at {completed}/{repetitions}", flush=True)
    started = time.perf_counter()
    for repetition in range(completed, repetitions):
        # Identical seed streams give nested prefixes across sample sizes.
        rng = np.random.default_rng(point_settings["base_seed"] + repetition)
        samples = rng.normal(0.0, np.sqrt(point_settings["sigma2"]), size=int(sample_size))
        divergences, mass_errors[repetition] = compute_n_sweep_divergences(
            samples, h_grid, m_x=point_settings["m_x"], padding=point_settings["padding"],
        )
        for column, name in enumerate(N_SWEEP_METHODS):
            indices[repetition, column] = spectral_choice_from_divergence(
                divergences[name], point_settings["lambda_value"]
            )["selected_index"]
        completed = repetition + 1
        save_checkpoint()
        if completed == 1 or completed % 25 == 0 or completed == repetitions:
            print(f"n={sample_size}: {completed}/{repetitions}; elapsed this run: "
                  f"{time.perf_counter() - started:.1f}s", flush=True)

    _validate_n_sweep_checkpoint(indices, mass_errors, completed, repetitions, len(h_grid))
    counts = {name: np.bincount(indices[:, column], minlength=len(h_grid))
              for column, name in enumerate(N_SWEEP_METHODS)}
    frequencies = {name: values / repetitions for name, values in counts.items()}
    for name in N_SWEEP_METHODS:
        assert counts[name].sum() == repetitions
        assert np.isclose(frequencies[name].sum(), 1.0)
    return {
        "sample_size": int(sample_size), "settings": point_settings,
        "selected_indices": indices, "counts": counts, "frequencies": frequencies,
        "maximum_grid_mass_error": float(np.max(mass_errors)),
        "directory": point_directory, "source_checkpoint": source_checkpoint,
    }


In [ ]:
n_sweep_results = {}
for current_n in N_SWEEP_SAMPLE_SIZES:
    n_sweep_results[current_n] = run_n_sweep_point(
        current_n, n_sweep_base_settings, n_sweep_h_grid, n_sweep_run_directory,
        existing_data_directory=data_output_directory, profile=N_SWEEP_PROFILE,
        reuse=N_SWEEP_REUSE_CHECKPOINTS,
    )
print(f"Completed {len(n_sweep_results)} sample sizes and "
      f"{len(N_SWEEP_METHODS) * len(n_sweep_results)} bandwidth distributions.")


In [ ]:
n_sweep_selection_rows = []
n_sweep_frequency_rows = []
n_sweep_summary_rows = []
for current_n in N_SWEEP_SAMPLE_SIZES:
    result = n_sweep_results[current_n]
    for column, name in enumerate(N_SWEEP_METHODS):
        chosen_indices = result["selected_indices"][:, column]
        chosen_h = n_sweep_h_grid[chosen_indices]
        n_sweep_selection_rows.extend(
            {"n": current_n, "repetition": repetition,
             "seed": n_sweep_base_settings["base_seed"] + repetition,
             "method": name, "grid_index": int(index), "h": float(n_sweep_h_grid[index])}
            for repetition, index in enumerate(chosen_indices)
        )
        n_sweep_frequency_rows.extend(
            {"n": current_n, "method": name, "grid_index": index, "h": float(h),
             "count": int(result["counts"][name][index]),
             "relative_frequency": float(result["frequencies"][name][index])}
            for index, h in enumerate(n_sweep_h_grid)
        )
        modal_index = int(np.argmax(result["counts"][name]))
        n_sweep_summary_rows.append({
            "n": current_n, "method": name,
            "repetitions": n_sweep_base_settings["repetitions"],
            "modal_h": float(n_sweep_h_grid[modal_index]),
            "mean_h": float(np.mean(chosen_h)), "median_h": float(np.median(chosen_h)),
            "modal_frequency": float(result["frequencies"][name][modal_index]),
            "maximum_grid_mass_error": result["maximum_grid_mass_error"],
        })
for filename, rows in (
    ("selections.csv", n_sweep_selection_rows),
    ("frequencies.csv", n_sweep_frequency_rows),
    ("summary.csv", n_sweep_summary_rows),
):
    with (n_sweep_run_directory / filename).open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
print(f"Saved selections, all 30 distributions, and summary: {n_sweep_run_directory}")
print(f"{'n':>5s} {'method':8s} {'modal h':>10s} {'mean h':>10s} {'median h':>10s}")
for row in n_sweep_summary_rows:
    print(f"{row['n']:5d} {row['method']:8s} {row['modal_h']:10.5f} "
          f"{row['mean_h']:10.5f} {row['median_h']:10.5f}")


### Ten separate distribution panels per divergence

Run this cell after the sweep has finished. TV, H², and JS each receive a separate 5 × 2 panel figure. Sample sizes increase from left to right and then down. The common axis limits make all 30 distributions directly comparable. When enabled above, the dashed line in each panel is the Gaussian **MISE reference**, not a TV-, H²-, or JS-optimal bandwidth.

Source for the optional reference: [Gaussian MISE helper](kde_tools/bandwidths.py).


In [ ]:
from matplotlib.lines import Line2D

if set(n_sweep_results) != set(N_SWEEP_SAMPLE_SIZES):
    raise RuntimeError("Finish all ten sample sizes before plotting the distributions.")
n_sweep_positions = np.arange(len(n_sweep_h_grid))
n_sweep_ticks = np.unique(np.linspace(0, len(n_sweep_h_grid) - 1, 7, dtype=int))
n_sweep_y_max = 1.08 * max(
    np.max(n_sweep_results[current_n]["frequencies"][name])
    for current_n in N_SWEEP_SAMPLE_SIZES for name in N_SWEEP_METHODS
)
n_sweep_colors = {"TV": "#0072B2", "H2": "#49A994", "JS": "#C44E52"}
n_sweep_labels = {"TV": "TV", "H2": r"$H^2$", "JS": "JS"}

for name in N_SWEEP_METHODS:
    sweep_fig, sweep_axes = plt.subplots(
        5, 2, figsize=(14, 18), sharex=True, sharey=True,
    )
    for sweep_ax, current_n in zip(sweep_axes.flat, N_SWEEP_SAMPLE_SIZES):
        sweep_ax.bar(
            n_sweep_positions, n_sweep_results[current_n]["frequencies"][name],
            width=0.86, color=n_sweep_colors[name], alpha=0.85,
            edgecolor="white", linewidth=0.25,
        )
        if N_SWEEP_SHOW_MISE_REFERENCE:
            reference_h = np.sqrt(n_sweep_base_settings["sigma2"]) * gaussian_mise_oracle_bandwidth(current_n)
            if n_sweep_h_grid[0] <= reference_h <= n_sweep_h_grid[-1]:
                reference_position = np.interp(
                    np.log(reference_h), np.log(n_sweep_h_grid), n_sweep_positions,
                )
                sweep_ax.axvline(reference_position, color="#1f77b4", linestyle="--", linewidth=1.4)
        sweep_ax.set_title(rf"$n={current_n}$", fontsize=13, pad=8)
        sweep_ax.set_xticks(n_sweep_ticks, [f"{n_sweep_h_grid[index]:.3g}" for index in n_sweep_ticks])
        sweep_ax.set_xlim(-0.7, len(n_sweep_h_grid) - 0.3)
        sweep_ax.set_ylim(0, n_sweep_y_max)
        sweep_ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
        sweep_ax.set_xlabel("selected bandwidth h", fontsize=10)
        sweep_ax.set_ylabel("relative frequency", fontsize=10)
        sweep_ax.tick_params(labelsize=9, colors="#304967", labelbottom=True, labelleft=True)
        sweep_ax.grid(axis="y", color="#DDE5EC", linewidth=0.7)
        sweep_ax.set_axisbelow(True)
        for side in ("top", "right"):
            sweep_ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            sweep_ax.spines[side].set_color("#B8C5D1")
    sweep_fig.suptitle(
        f"Bandwidth-choice distributions for increasing sample size: {n_sweep_labels[name]}\n"
        + rf"Gaussian data; $M={n_sweep_base_settings['repetitions']}$ per $n$, "
        + rf"$N={len(n_sweep_h_grid)}$, $S=\exp(-\lambda D)$, "
        + rf"$\lambda={n_sweep_base_settings['lambda_value']:g}$ ({N_SWEEP_PROFILE})",
        fontsize=16, y=0.995,
    )
    if N_SWEEP_SHOW_MISE_REFERENCE:
        sweep_fig.legend(
            handles=[Line2D([0], [0], color="#1f77b4", linestyle="--", linewidth=1.4)],
            labels=["Gaussian MISE reference for each sample size"],
            loc="lower center", frameon=False, fontsize=11, bbox_to_anchor=(0.5, 0.005),
        )
    sweep_fig.tight_layout(rect=(0, 0.035, 1, 0.955), h_pad=2.0, w_pad=2.0)
    sweep_png_path = n_sweep_image_directory / (
        f"increasing_n_{name.lower()}_M{n_sweep_base_settings['repetitions']}_N{len(n_sweep_h_grid)}.png"
    )
    sweep_fig.savefig(sweep_png_path, dpi=160, bbox_inches="tight")
    plt.show()  # The shared automatic exporter saves the full ten-panel figure as a PDF.
    plt.close(sweep_fig)
    print(f"PNG ({name}): {sweep_png_path}")
print(f"PDF directory: {pdf_output_directory}")
